In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [2]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from string import punctuation

sw_indo = stopwords.words('indonesian') + list(punctuation)

# Import data

In [3]:
df = pd.read_csv('spam.csv')
df.head()

,Teks,label
0,[PROMO] Beli paket Flash mulai 1GB di MY TELKO...,1
1,2.5 GB/30 hari hanya Rp 35 Ribu Spesial buat A...,1
2,"2016-07-08 11:47:11.Plg Yth, sisa kuota Flash ...",1
3,"2016-08-07 11:29:47.Plg Yth, sisa kuota Flash ...",1
4,4.5GB/30 hari hanya Rp 55 Ribu Spesial buat an...,1


# dataset splitting

In [4]:
# values langsung convert ke array(np)
x = df.Teks
y = df.label

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, stratify=y, random_state=42)
x_train.shape, x_test.shape, y_train.shape, y_test.shape

((914,), (229,), (914,), (229,))

# training

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import RandomizedSearchCV


In [6]:
rsp_logreg_params = {
    'prep__max_df': [0.8, 0.9, 1.0],  
    'prep__min_df': [1, 5, 10],      
    'prep__ngram_range': [(1, 1), (1, 2), (1, 3)],
    'algo__C': np.logspace(-3, 2, 6), 
    'algo__penalty': ['l1', 'l2'],    
    'algo__solver': ['liblinear'] 
}

pipeline = Pipeline([
    ('prep', TfidfVectorizer(tokenizer=word_tokenize, stop_words=sw_indo)),
    ('algo', LogisticRegression(solver='lbfgs', n_jobs=-1, random_state=42))
])

# cv = cross validation
model = RandomizedSearchCV(pipeline, rsp_logreg_params, cv=3, n_iter=50,  n_jobs=-1, verbose=1)
model.fit(x_train, y_train)

print(model.best_params_)
print(model.score(x_train, y_train), model.best_score_, model.score(x_test, y_test))

Fitting 3 folds for each of 50 candidates, totalling 150 fits


c:\Users\akmal\anaconda3\envs\keras_env\lib\site-packages\sklearn\feature_extraction\text.py:525: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\akmal\anaconda3\envs\keras_env\lib\site-packages\sklearn\feature_extraction\text.py:408: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['``'] not in stop_words.
  warnings.warn(
c:\Users\akmal\anaconda3\envs\keras_env\lib\site-packages\sklearn\linear_model\_logistic.py:1222: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 16.
  warnings.warn(


{'prep__ngram_range': (1, 2), 'prep__min_df': 1, 'prep__max_df': 0.8, 'algo__solver': 'liblinear', 'algo__penalty': 'l2', 'algo__C': 100.0}
1.0 0.9627947943629566 0.9737991266375546


# sanity check

In [7]:
from sklearn.metrics import accuracy_score

score = model.predict(x_test)
accuracy = accuracy_score(y_test, score)
accuracy

0.9737991266375546

In [9]:
x_train

597                    Ada teh dikpad, ini kontaknya ####
344     km dari BANK BRI Sdh tlp tdk bs jd km SMS No.P...
930     Mohon maaf pak semalam saya Sudah konfirmasi t...
590                                Ada diruangan nya tadi
611                                 Aku depan pasca skrng
                              ...                        
433     Pesta m-tronik Anda sebagai Pelanggan setia m-...
144     Nikmati Double Internetan di jaringan Data ter...
872     Kan ini karna bulan baru qaqa 😂 bisa ngomong g...
670         Besok saya ada rapat dinas.. Selasa aja ya...
1012    Rumah aku Gais, lewat pager garasi, karna page...
Name: Teks, Length: 914, dtype: object

In [10]:
import mlflow
from mlflow.models import infer_signature

mlflow.set_tracking_uri(uri="http://127.0.0.1:5000")

mlflow.set_experiment("MLflow SMS Spam")

with mlflow.start_run():
    mlflow.log_params(rsp_logreg_params)

    mlflow.log_metric('accuracy', accuracy)

    mlflow.set_tag('Training info', 'LR model for SMS spam')

    x_train_df = x_train.to_frame(name='text_column')

    signature = infer_signature(x_train_df, model.predict(x_train))

    model_info = mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path='SMS_model',
        signature=signature,
        input_example=x_train_df,
        registered_model_name='tracking-quickstart'
    )



Successfully registered model 'tracking-quickstart'.
2025/06/10 19:37:10 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: tracking-quickstart, version 1
Created version '1' of model 'tracking-quickstart'.


c:\Users\akmal\anaconda3\envs\keras_env\lib\site-packages\sklearn\feature_extraction\text.py:408: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['``'] not in stop_words.
  warnings.warn(
2025/06/10 19:37:11 INFO mlflow.tracking._tracking_service.client: 🏃 View run serious-doe-310 at: http://127.0.0.1:5000/#/experiments/790102350447156196/runs/3dbffa0e4c3047db93483a7b87d288a1.
2025/06/10 19:37:11 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/790102350447156196.
